In [3]:
!git clone https://github.com/bgshih/coco-text.git

fatal: destination path 'coco-text' already exists and is not an empty directory.


In [18]:
# Import Python Standard Library dependencies
import datetime
from functools import partial
from glob import glob
import json
import math
import multiprocessing
import os
from pathlib import Path
import random
from typing import Any, Dict, Optional




# Import matplotlib for creating plots
import matplotlib.pyplot as plt

# Import numpy
import numpy as np

# Import the pandas package
import pandas as pd

# Set options for Pandas DataFrame display
pd.set_option('max_colwidth', None)  # Do not truncate the contents of cells in the DataFrame
pd.set_option('display.max_rows', None)  # Display all rows in the DataFrame
pd.set_option('display.max_columns', None)  # Display all columns in the DataFrame

# Import PIL for image manipulation
from PIL import Image, ImageDraw

# Import PyTorch dependencies
import torch
from torch.amp import autocast
from torch.cuda.amp import GradScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.tv_tensors import BoundingBoxes, Mask
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
import torchvision.transforms.v2  as transforms
from torchvision.transforms.v2 import functional as TF

# Import Mask R-CNN
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


import sys
sys.path.append('./coco-text')
# import coco_text

In [21]:
# Set the seed for generating random numbers in PyTorch, NumPy, and Python's random module.
seed = 1234
device = get_torch_device()
dtype = torch.float32
device, dtype

NameError: name 'set_seed' is not defined

In [23]:
model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
model.eval()
summary(model, input_size=(3, 800, 800))

AttributeError: 'ImageList' object has no attribute 'size'

In [13]:
total_params = sum(p.numel() for p in model.parameters())
print(f'Total number of parameters: {total_params}')

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Number of trainable parameters: {trainable_params}')

Total number of parameters: 46359409
Number of trainable parameters: 46134065


In [6]:
# The name for the project
project_name = f"project-mask-rcnn-ocr"

# The path for the project folder
project_dir = Path(f"./{project_name}/")

# Create the project directory if it does not already exist
project_dir.mkdir(parents=True, exist_ok=True)

# Define path to store datasets
dataset_dir = Path("./Datasets/")
# Create the dataset directory if it does not exist
dataset_dir.mkdir(parents=True, exist_ok=True)

pd.Series({
    "Project Directory:": project_dir, 
    "Dataset Directory:": dataset_dir
}).to_frame().style.hide(axis='columns')

Project Directory:,project-mask-rcnn-ocr
Dataset Directory:,Datasets


In [7]:
# Create the path to the directory where the dataset will be extracted
dataset_path = Path(f'{dataset_dir}/train2014')

# Get a list of image files in the dataset
img_file_paths = get_img_files(dataset_path)

json_file_path = f'./{dataset_dir}/cocotext.v2.json'

# Display the names of the folders using a Pandas DataFrame
pd.DataFrame({"Image File": [file.name for file in img_file_paths]}).head()
json_file_path

'./Datasets/cocotext.v2.json'

In [8]:
ct = coco_text.COCO_Text(json_file_path)


loading annotations into memory...
0:00:00.941641
creating index...
index created!


In [9]:
with open(json_file_path, 'r') as f:
    data = json.load(f)

In [10]:
for item in data['cats'].items():
    print(item)
    break

In [13]:
# # Load the data from the cocotext.v2.json file
# with open(f'{dataset_dir}/cocotext.v2.json', 'r') as f:
#     data = json.load(f)
# # ct = coco_text.COCO_Text(f'{dataset_dir}/cocotext.v2.json')
# Initialize an empty list to store the dictionaries
image_annotations = []
file_set = set(file.name for file in img_file_paths)

# Iterate over the annotations in the data
for img_id, annotation in ct.imgs.items():
    # Create a dictionary with the image file name and the annotation
    # Iterate over the annotations in img_file_paths
    if annotation['file_name'] in file_set:
        fn = annotation['file_name']

    image_annotation = {"file_name":fn,
                        "image_iD": img_id, "annotation": annotation}


    # Append the dictionary to the list
    image_annotations.append(image_annotation)


# Convert the list of dictionaries into a pandas DataFrame
annotation_df = pd.DataFrame(image_annotations)

# annotation_df['index'] = annotation_df.apply(lambda row: row['imagePath'].split('.')[0], axis=1)
# annotation_df = annotation_df.set_index('index')
# annotation_df['index'] = 
# Display the first few rows of the DataFrame
annotation_df.head()

,file_name,image_iD,annotation
0,COCO_train2014_000000540965.jpg,540965,"{'id': 540965, 'set': 'train', 'width': 640, 'file_name': 'COCO_train2014_000000540965.jpg', 'height': 360}"
1,COCO_train2014_000000229078.jpg,229078,"{'id': 229078, 'set': 'train', 'width': 640, 'file_name': 'COCO_train2014_000000229078.jpg', 'height': 480}"
2,COCO_train2014_000000260932.jpg,260932,"{'id': 260932, 'set': 'train', 'width': 640, 'file_name': 'COCO_train2014_000000260932.jpg', 'height': 512}"
3,COCO_train2014_000000103812.jpg,103812,"{'id': 103812, 'set': 'train', 'width': 640, 'file_name': 'COCO_train2014_000000103812.jpg', 'height': 427}"
4,COCO_train2014_000000304937.jpg,304937,"{'id': 304937, 'set': 'train', 'width': 640, 'file_name': 'COCO_train2014_000000304937.jpg', 'height': 640}"


In [ ]:
with open(json_file_path, 'r') as json_file:
    annotation_df = pd.read_json(json_file)

# Assign the image file name as the index for each row
# annotation_df['index'] = annotation_df.apply(lambda row: row['file_name'].split('.')[0], axis=1)
# annotation_df = annotation_df.set_index('index')

annotation_df.head()

,cats,anns,imgs,imgToAnns,info
45346,NaN,"{'mask': [468.9, 286.7, 468.9, 295.2, 493.0, 295.8, 493.0, 287.2], 'class': 'machine printed', 'bbox': [468.9, 286.7, 24.1, 9.1], 'image_id': 217925, 'id': 45346, 'language': 'english', 'area': 206.06, 'utf8_string': 'New', 'legibility': 'legible'}",NaN,NaN,NaN
153036,NaN,"{'mask': [344.5, 261.5, 348.1, 261.5, 348.2, 263.4, 344.5, 263.4], 'class': 'machine printed', 'bbox': [344.5, 261.5, 3.7, 1.9000000000000001], 'image_id': 483569, 'id': 153036, 'language': 'english', 'area': 6.93, 'utf8_string': '', 'legibility': 'illegible'}",NaN,NaN,NaN
125303,NaN,"{'mask': [362.4, 280.9, 359.2, 286.3, 367.1, 291.0, 369.9, 285.0], 'class': 'machine printed', 'bbox': [359.2, 280.9, 10.7, 10.1], 'image_id': 417153, 'id': 125303, 'language': 'english', 'area': 57.09, 'utf8_string': '', 'legibility': 'illegible'}",NaN,NaN,NaN
21639,NaN,"{'mask': [570.9, 9.6, 570.9, 14.3, 557.1, 14.3, 556.8, 9.4], 'class': 'machine printed', 'bbox': [556.8, 9.4, 14.1, 4.9], 'image_id': 15451, 'id': 21639, 'language': 'english', 'area': 66.94, 'utf8_string': '', 'legibility': 'illegible'}",NaN,NaN,NaN
112792,NaN,"{'mask': [489.4, 131.8, 502.7, 135.6, 502.7, 156.4, 493.1, 155.3], 'class': 'machine printed', 'bbox': [489.4, 131.8, 13.3, 24.6], 'image_id': 379024, 'id': 112792, 'language': 'english', 'area': 249.08, 'utf8_string': 'W', 'legibility': 'legible'}",NaN,NaN,NaN


In [ ]:
ct.anns

{45346: {'mask': [468.9, 286.7, 468.9, 295.2, 493.0, 295.8, 493.0, 287.2],
  'class': 'machine printed',
  'bbox': [468.9, 286.7, 24.1, 9.1],
  'image_id': 217925,
  'id': 45346,
  'language': 'english',
  'area': 206.06,
  'utf8_string': 'New',
  'legibility': 'legible'},
 153036: {'mask': [344.5, 261.5, 348.1, 261.5, 348.2, 263.4, 344.5, 263.4],
  'class': 'machine printed',
  'bbox': [344.5, 261.5, 3.7, 1.9],
  'image_id': 483569,
  'id': 153036,
  'language': 'english',
  'area': 6.93,
  'utf8_string': '',
  'legibility': 'illegible'},
 125303: {'mask': [362.4, 280.9, 359.2, 286.3, 367.1, 291.0, 369.9, 285.0],
  'class': 'machine printed',
  'bbox': [359.2, 280.9, 10.7, 10.1],
  'image_id': 417153,
  'id': 125303,
  'language': 'english',
  'area': 57.09,
  'utf8_string': '',
  'legibility': 'illegible'},
 21639: {'mask': [570.9, 9.6, 570.9, 14.3, 557.1, 14.3, 556.8, 9.4],
  'class': 'machine printed',
  'bbox': [556.8, 9.4, 14.1, 4.9],
  'image_id': 15451,
  'id': 21639,
  'langua